# TalentIQ - Exploratory Data Analysis

This notebook explores the IBM HR Employee Attrition dataset.
The goal is to understand the data before building any model.

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print('Dataset loaded:', df.shape)

---

## Step 1 - What is the business problem?

**Problem:**
A company loses employees every year. Hiring a replacement costs time and money.
The company wants to know in advance which employees are likely to leave.
If they can predict it early, they can take action to retain that person.

**What we are solving:**
We need to build a model that looks at an employee's profile (age, salary, job role, etc.)
and predicts whether they will leave (Attrition = Yes) or stay (Attrition = No).

This is a binary classification problem.
- Label 1 = Employee left (Attrition = Yes)
- Label 0 = Employee stayed (Attrition = No)

**Solution approach:**
Before building any model, we first need to understand the data.
This notebook answers the following questions:
1. What does the data look like?
2. Are there any missing values or duplicate rows?
3. How many employees left vs stayed? Is the data imbalanced?
4. Which features seem to matter for attrition?
5. Are there any outliers?
6. How are the features related to each other?

---

## Step 2 - What does the data look like?

**Problem:** We have a CSV file but we do not know what columns it has, how many rows, and what the data looks like.

**Solution:** Load the data and take a first look at its shape, column names, and sample rows.

In [ ]:
print('Shape:', df.shape)
print('Columns:', list(df.columns))

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

**What we found:**
- The dataset has 1470 rows and 35 columns.
- Some columns are numbers (Age, MonthlyIncome) and some are text (Department, Gender).
- The target column is Attrition which has values Yes or No.

---

## Step 3 - Are there missing values or duplicates?

**Problem:** If the data has missing values, our model will break or give wrong results.
Duplicate rows can also trick the model into learning the same pattern multiple times.

**Solution:** Check for both and report the counts.

In [ ]:
print('Missing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nTotal duplicate rows:', df.duplicated().sum())

**What we found:**
- No missing values in this dataset. Every column is complete.
- No duplicate rows either.
- This means we can move forward without heavy cleaning.

There are a few columns that carry no information and will be dropped later:
- EmployeeCount: always 1 for every row
- StandardHours: always 80 for every row
- Over18: always Y for every row
- EmployeeNumber: just an ID, not useful for prediction

---

## Step 4 - Is the target balanced?

**Problem:** If 95% of employees stayed and only 5% left, a model that always predicts "stay" would be 95% accurate but completely useless.
This is called class imbalance and it must be checked before training.

**Solution:** Count how many employees left vs stayed and visualize it.

In [ ]:
counts = df['Attrition'].value_counts()
pct = df['Attrition'].value_counts(normalize=True) * 100

print('Attrition Counts:')
print(counts)
print('\nPercentage:')
print(pct.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'])
ax.set_title('Class Distribution - Attrition')
ax.set_xlabel('Attrition')
ax.set_ylabel('Count')
ax.set_xticklabels(['No (Stayed)', 'Yes (Left)'], rotation=0)
for i, v in enumerate(counts):
    ax.text(i, v + 5, str(v), ha='center')
plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=150)
plt.show()

**What we found:**
- About 84% of employees stayed and 16% left.
- This is a moderately imbalanced dataset.
- If we use accuracy as our metric, it can be misleading.
- We will use F1-macro as our main evaluation metric.
- We will also apply SMOTE during training to generate synthetic samples for the minority class.

---

## Step 5 - Which features matter for attrition?

**Problem:** We have 35 columns. We need to understand which ones are actually related to whether an employee leaves or stays.
Looking at all columns randomly would take too long and would not tell us anything useful.

**Solution:** Split features into numerical and categorical. For numerical, compare distributions between the two groups. For categorical, check attrition rates per category.

---

### Numerical Features

In [ ]:
numerical_cols = ['Age', 'MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany',
                  'YearsWithCurrManager', 'DistanceFromHome', 'NumCompaniesWorked',
                  'YearsSinceLastPromotion', 'PercentSalaryHike']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    for label, color in zip(['No', 'Yes'], ['steelblue', 'tomato']):
        axes[i].hist(df[df['Attrition'] == label][col], bins=20,
                     alpha=0.6, color=color, label=label)
    axes[i].set_title(col)
    axes[i].legend(title='Left')

plt.suptitle('Numerical Feature Distributions by Attrition', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/numerical_distributions.png', dpi=150)
plt.show()

**What we found from numerical features:**
- Younger employees are more likely to leave.
- Employees with lower monthly income tend to leave more.
- Employees with fewer total working years and fewer years at the company leave more.
- Distance from home has some effect but it is weaker.
- Employees who have worked at more companies before also tend to leave more.

---

### Categorical Features

In [ ]:
categorical_cols = ['Department', 'JobRole', 'MaritalStatus', 'OverTime',
                    'BusinessTravel', 'Gender', 'EducationField']

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    attrition_rate = df.groupby(col)['Attrition'].apply(
        lambda x: (x == 'Yes').mean() * 100
    ).sort_values(ascending=False)
    attrition_rate.plot(kind='bar', ax=axes[i], color='tomato')
    axes[i].set_title(f'Attrition Rate by {col}')
    axes[i].set_ylabel('Attrition Rate (%)')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=30, ha='right')

fig.delaxes(axes[-1])
plt.tight_layout()
plt.savefig('../reports/figures/categorical_attrition_rates.png', dpi=150)
plt.show()

**What we found from categorical features:**
- OverTime is the strongest signal. Employees who do overtime leave at a much higher rate.
- Single employees leave more than married or divorced ones.
- Sales Representatives have the highest attrition rate among all job roles.
- Frequent travelers leave more than non-travelers.
- Gender has a small effect and is weaker compared to other features.

---

## Step 6 - Are there outliers in numerical columns?

**Problem:** Extreme values (outliers) can distort what the model learns.
For example, one employee earning 10x more than everyone else can skew the model toward income.

**Solution:** Use box plots to spot outliers visually.

In [ ]:
outlier_cols = ['MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany', 'DistanceFromHome']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for i, col in enumerate(outlier_cols):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)

plt.suptitle('Outlier Check - Box Plots', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/boxplots.png', dpi=150)
plt.show()

**What we found:**
- MonthlyIncome and TotalWorkingYears both have visible outliers on the high end.
- YearsAtCompany also has some employees with very long tenure.
- These will be clipped using the IQR method during preprocessing to prevent them from distorting the model.

---

## Step 7 - How are numerical features related to each other?

**Problem:** If two features are highly correlated (for example, Age and TotalWorkingYears), they carry the same information.
This is called multicollinearity and it can confuse some models.

**Solution:** Create a correlation heatmap to see how strongly each pair of numerical features is related.

In [ ]:
num_df = df.select_dtypes(include='number').drop(
    columns=['EmployeeCount', 'StandardHours', 'EmployeeNumber'], errors='ignore'
)

# Map Attrition to numeric for correlation
num_df['Attrition'] = (df['Attrition'] == 'Yes').astype(int)

corr = num_df.corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, annot_kws={'size': 7})
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('../reports/figures/heatmap.png', dpi=150)
plt.show()

In [ ]:
# Show top correlations with the target
target_corr = corr['Attrition'].drop('Attrition').sort_values(key=abs, ascending=False)
print('Top features correlated with Attrition:')
print(target_corr.head(10).round(3))

**What we found:**
- Age, TotalWorkingYears, YearsAtCompany, and YearsWithCurrManager are strongly correlated with each other. This is expected since they all measure time and experience.
- MonthlyIncome is also correlated with Age and TotalWorkingYears. Older and more experienced employees earn more.
- The features most correlated with Attrition are: YearsWithCurrManager, TotalWorkingYears, YearsAtCompany, and MonthlyIncome (all negative, meaning more years / more income = less likely to leave).

---

## Summary - What did EDA tell us?

| Finding | Action Taken |
|---|---|
| No missing values | No imputation needed |
| No duplicates | No rows removed at this stage |
| Target is imbalanced (84% vs 16%) | Use F1-macro as metric, apply SMOTE during training |
| OverTime is a strong predictor | Keep as feature |
| MonthlyIncome, Age, Years columns matter | Keep and engineer new features from them |
| Outliers in MonthlyIncome, TotalWorkingYears, etc. | Clip using IQR in preprocessing |
| Correlated time-based features | Engineer combined features like LoyaltyIndex, CareerGrowthRate |
| Useless constant columns (EmployeeCount, StandardHours, Over18) | Drop during preprocessing |

The next step is preprocessing followed by feature engineering, where we act on all of these findings.